# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Load Plotly's built-in Gapminder dataset and filter to 2007
gm = px.data.gapminder()
gm_2007 = gm.loc[gm["year"] == 2007].copy()

# Calculate the global mean and each country's deviation from it
global_mean_life_exp = gm_2007["lifeExp"].mean()
gm_2007["lifeExp_deviation"] = gm_2007["lifeExp"] - global_mean_life_exp

# Find the continent with the lowest mean life expectancy
continent_means = (
    gm_2007.groupby("continent")["lifeExp"]
    .mean()
    .sort_values()
)
lowest_continent = continent_means.index[0]
lowest_continent_mean = continent_means.iloc[0]

print(f"Countries included: {len(gm_2007)}")
print(f"Global mean life expectancy (2007): {global_mean_life_exp:.2f} years")
print(
    f"Lowest continental average: {lowest_continent} "
    f"({lowest_continent_mean:.2f} years)"
)

gm_2007[["country", "continent", "lifeExp", "lifeExp_deviation"]].head()


In [ ]:
fig1 = px.choropleth(
    gm_2007,
    locations="iso_alpha",
    locationmode="ISO-3",
    color="lifeExp_deviation",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    hover_name="country",
    hover_data={
        "lifeExp": ":.1f",
        "lifeExp_deviation": ":+.1f",
        "iso_alpha": False,
        "continent": True,
    },
    labels={
        "lifeExp": "Life expectancy (years)",
        "lifeExp_deviation": "Difference from global average (years)",
        "continent": "Continent",
    },
    title=(
        f"{lowest_continent} has the lowest average life expectancy in 2007 "
        f"— {lowest_continent_mean:.1f} years vs {global_mean_life_exp:.1f} globally"
    ),
    projection="natural earth",
    height=600,
)

fig1.update_layout(
    font=dict(family="Arial", size=12),
    geo=dict(showframe=False, showcoastlines=True),
    coloraxis_colorbar=dict(
        title="Years vs<br>global average",
        thickness=16,
        len=0.65,
    ),
    margin=dict(l=0, r=0, t=85, b=0),
)
fig1.show()

## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** *(URL or description)*

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

*Your answer here*

**Colour scale chosen** (sequential or diverging) **and reason:**

*Your answer here*


In [ ]:
import json
import pandas as pd
import plotly.express as px

URL = "https://gist.githubusercontent.com/jbrobst/56c13bbbf9d97d187fea01ca62ea5112/raw/e388c4cae20aa53cb5090210a42ebb9b765c0a36/india_states.geojson"

try:
    from pyodide.http import open_url          # browser notebook
    geojson = json.load(open_url(URL))
except ImportError:
    import urllib.request                      # Jupyter on your laptop
    geojson = json.load(urllib.request.urlopen(URL))

print(len(geojson['features']), "features")
print(geojson['features'][0]['properties'])
print(sorted(f['properties']['ST_NM'] for f in geojson['features']))

In [ ]:
literacy = {
    'Kerala': 95.1, 'Lakshadweep': 90.8, 'Mizoram': 93.2, 'Tripura': 89.1,
    'Goa': 88.6, 'Daman and Diu': 85.9, 'Puducherry': 87.4, 'Chandigarh': 89.0,
    'Delhi': 88.1, 'Andaman and Nicobar Islands': 84.7, 'Himachal Pradesh': 86.2, 'Maharashtra': 84.1,
    'Sikkim': 85.3, 'Tamil Nadu': 82.6, 'Nagaland': 81.7, 'Manipur': 83.5,
    'Uttarakhand': 82.8, 'Gujarat': 81.4, 'Dadra and Nagar Haveli': 79.2, 'West Bengal': 80.3,
    'Punjab': 82.1, 'Haryana': 81.0, 'Karnataka': 79.8, 'Meghalaya': 78.9,
    'Odisha': 77.6, 'Assam': 76.8, 'Chhattisgarh': 75.9, 'Madhya Pradesh': 74.3,
    'Uttar Pradesh': 73.8, 'Jammu and Kashmir': 77.1, 'Andhra Pradesh': 78.4, 'Jharkhand': 72.9,
    'Rajasthan': 74.8, 'Arunachal Pradesh': 79.5, 'Telangana': 80.7, 'Bihar': 70.6,
}
idf = pd.DataFrame({'state': list(literacy), 'literacy': list(literacy.values())})

# The GeoJSON spells some names differently, so map them to its spelling.
# Edit the right-hand side after checking the "Missing" output below.
rename = {
    'Andaman and Nicobar Islands': 'Andaman & Nicobar Island',
    'Jammu and Kashmir': 'Jammu & Kashmir',
    'Dadra and Nagar Haveli': 'Dadara & Nagar Havelli',
    'Daman and Diu': 'Daman & Diu',
    'Delhi': 'NCT of Delhi',
    'Arunachal Pradesh': 'Arunanchal Pradesh',
}
idf['state'] = idf['state'].replace(rename)

KEY = 'ST_NM'
geo_names = {f['properties'][KEY] for f in geojson['features']}
print("In my data but not in GeoJSON:", set(idf['state']) - geo_names)
print("In GeoJSON but not in my data:", geo_names - set(idf['state']))

In [ ]:
fig = px.choropleth(
    idf,
    geojson=geojson,
    locations='state',
    featureidkey=f'properties.{KEY}',
    color='literacy',
    color_continuous_scale='YlGnBu',
    labels={'literacy': 'Literacy rate (%)'},
    hover_name='state',
    template='plotly_white',
    title='Literacy peaks in Kerala and the north-east, and is lowest across Bihar and the central belt',
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_traces(marker_line_color='white', marker_line_width=0.6)
fig.update_layout(margin=dict(l=0, r=0, t=70, b=0), height=700,
                  title_x=0, coloraxis_colorbar=dict(title='Literacy<br>rate (%)'))
fig.show()